In [1]:
import builtins

# The Magic Hack: Create a dummy class and inject it into Python's builtins
# so the Python 3.12 type-hint evaluator finds it and stops crashing.
class DummyPeftConfig:
    pass

if not hasattr(builtins, "PeftConfigLike"):
    builtins.PeftConfigLike = DummyPeftConfig

import os
import sys
import json
import torch
import logging
from google.cloud import storage

# Configure logging
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

/opt/venv/lib/python3.12/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [19]:
# --- Configuration ---
# Start with the smallest model to verify code execution without OOM
MODEL_NAME = "openai/whisper-tiny"
# MODEL_NAME = "openai/whisper-large-v3-turbo"  # Options: "openai/whisper-large-v3", "openai/whisper-large-v3-turbo"

# Whisper models natively expect audio sampled at 16,000 Hz
TARGET_SAMPLE_RATE = 16000

# Dataset manifest paths.
# Can be local file paths (e.g. "/path/to/manifest.jsonl") or GCS URIs (e.g. "gs://bucket/manifest.jsonl")
TRAIN_MANIFEST_PATH = "gs://wd-transcription-data/segmented_audio/echo/eval/audio_raw/batch_manifest.jsonl"
VAL_MANIFEST_PATH = "gs://wd-transcription-data/segmented_audio/one_hour_pilot_audio_raw/batch_manifest.jsonl"

# PEFT configuration (highly recommended to avoid OOM on free colab GPUs)
USE_PEFT = False
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05

# TODO(varun): Review these parameters
# Training arguments
BATCH_SIZE = 4  # Reduce batch size (e.g. to 1 or 2) if you encounter OOM on GPU
GRADIENT_ACCUMULATION_STEPS = 4 # Simulates larger batch sizes without memory penalty
LEARNING_RATE = 1e-5
WARMUP_STEPS = 2
MAX_STEPS = 2 # Adjust to run more training steps
EVAL_STEPS = 3
SAVE_STEPS = 100
LOGGING_STEPS = 25

GCP_PROJECT_ID = "automatic-hawk-481415-m9"
OUTPUT_DIR = "../trained_checkpoints/"

In [10]:
# @title Audio & Dataset Helpers
import os
import json
import urllib.parse
import torch
import torchaudio
import torchaudio.transforms as T
import soundfile as sf
from torch.utils.data import Dataset

def get_local_audio_path(audio_filepath, storage_client=None, cache_dir="/tmp/audio_cache"):
    """Resolves audio file path. If it is a GCS path, downloads and caches it locally."""
    os.makedirs(cache_dir, exist_ok=True)
    if audio_filepath.startswith("gs://"):
        if storage_client is None:
            raise ValueError("storage_client must be provided to download GCS paths")
        
        # Parse GCS URI
        parsed = urllib.parse.urlparse(audio_filepath)
        bucket_name = parsed.netloc
        blob_name = parsed.path.lstrip('/')
        
        # Create a safe local file name using bucket and path to avoid collision
        safe_name = f"{bucket_name}_{blob_name.replace('/', '_')}"
        local_path = os.path.join(cache_dir, safe_name)
        
        if not os.path.exists(local_path):
            print(f"Downloading {audio_filepath} to local cache...")
            bucket = storage_client.bucket(bucket_name)
            blob = bucket.blob(blob_name)
            blob.download_to_filename(local_path)
        return local_path
    else:
        return audio_filepath

def load_audio_segment(filepath, offset=0.0, duration=None, target_sr=TARGET_SAMPLE_RATE):
    """Loads a segment of an audio file, downmixes it to mono, and resamples it to target_sr."""
    try:
        # Read audio metadata first to obtain sample rate using soundfile
        info = sf.info(filepath)
        sr = info.samplerate
        
        # Calculate frame offset and count based on original sample rate
        frame_offset = int(offset * sr) if offset is not None else 0
        num_frames = int(duration * sr) if duration is not None else -1
        
        if frame_offset < 0:
            frame_offset = 0
        if num_frames < 0:
            num_frames = -1
            
        waveform, sample_rate = torchaudio.load(
            filepath, 
            frame_offset=frame_offset, 
            num_frames=num_frames
        )
        
        # Average to mono
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
            
        # Resample to target_sr
        if sample_rate != target_sr:
            resampler = T.Resample(orig_freq=sample_rate, new_freq=target_sr)
            waveform = resampler(waveform)
            
        return waveform.squeeze(0)
    except Exception as e:
        logger.error(f"Error loading audio segment from {filepath} (offset: {offset}, duration: {duration}): {e}")
        raise e

class NemoDataset(Dataset):
    """Custom Dataset for NeMo manifests."""
    def __init__(self, manifest_path_or_uri, processor, storage_client=None, cache_dir="/tmp/audio_cache"):
        self.entries = []
        self.processor = processor
        self.storage_client = storage_client
        self.cache_dir = cache_dir
        
        # Resolve manifest path (handles GCS and local files)
        if manifest_path_or_uri.startswith("gs://"):
            if storage_client is None:
                raise ValueError("storage_client must be provided to download GCS manifest")
            # Download manifest
            parsed = urllib.parse.urlparse(manifest_path_or_uri)
            bucket = storage_client.bucket(parsed.netloc)
            blob = bucket.blob(parsed.path.lstrip('/'))
            
            manifest_content = blob.download_as_text()
            lines = manifest_content.strip().split('\n')
        else:
            with open(manifest_path_or_uri, 'r', encoding='utf-8') as f:
                lines = f.readlines()
                
        for line in lines:
            if line.strip():
                self.entries.append(json.loads(line))
                
        print(f"Loaded {len(self.entries)} entries from manifest: {manifest_path_or_uri}")

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        entry = self.entries[idx]
        audio_filepath = entry["audio_filepath"]
        text = entry["text"]
        # Manually hacking this because we are using batch manifests which currently
        # still contain the offset and duration of the original segment (before
        # clipping the audio). We will fix these manifests to relabel the 'offset'
        # and 'duration' fields to 'original_offset' and 'original_duration', and we
        # can then remove this hack.
        offset = 0.0
        duration = None
        
        # 1. Get local path to the audio file (handles on-demand GCS download)
        local_path = get_local_audio_path(audio_filepath, self.storage_client, self.cache_dir)
        
        # 2. Load the audio segment at target sample rate
        waveform = load_audio_segment(local_path, offset, duration, target_sr=TARGET_SAMPLE_RATE)
        
        # 3. Extract input_features
        # Note: Whisper processor handles standard padding to 30s
        inputs = self.processor(audio=waveform.numpy(), sampling_rate=TARGET_SAMPLE_RATE)
        input_features = inputs.input_features[0]
        
        # 4. Tokenize transcription
        labels = self.processor.tokenizer(text).input_ids
        
        return {
            "input_features": input_features,
            "labels": labels
        }

In [11]:
# @title Initialize Model, Processor, and PEFT/LoRA
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

print(f"Loading processor for {MODEL_NAME}...")
# Load processor (combines feature extractor and tokenizer)
processor = WhisperProcessor.from_pretrained(
    MODEL_NAME,
    language="english",
    task="transcribe"
)

print(f"Loading model {MODEL_NAME}...")
# Load Whisper model
model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    # This is causig problems, experimenting.
    # torch_dtype=torch.float16,
    torch_dtype=torch.float32,
    device_map="auto"
)

# Override configuration parameters
# TODO(varun): The agent added these configs, but I didn't quite understand
# its explanation when I asked what these do, needs more background understanding
# of Whisper and how it uses some specific tokens during training.
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

if USE_PEFT:
    print("Configuring and applying PEFT/LoRA...")
    # Prepare model for FP16 training with gradient checkpointing
    model.gradient_checkpointing_enable()
    
    # TODO(varun): Look at the whisper architecture carefully to see whether
    # these q_proj and v_proj matrices cover the audio encoding process as
    # well.
    peft_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=["q_proj", "v_proj"], # Whisper's multi-head attention modules
        lora_dropout=LORA_DROPOUT,
        bias="none",
        task_type="SEQ_2_SEQ_LM"
    )
    model = get_peft_model(model, peft_config)
    model.print_trainable_parameters()
else:
    print("PEFT disabled. Training full parameters.")

2026-05-14 11:26:06,016 - INFO - HTTP Request: GET https://huggingface.co/api/models/openai/whisper-tiny/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"


Loading processor for openai/whisper-tiny...


2026-05-14 11:26:06,124 - INFO - HTTP Request: HEAD https://huggingface.co/openai/whisper-tiny/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-05-14 11:26:06,234 - INFO - HTTP Request: HEAD https://huggingface.co/openai/whisper-tiny/resolve/main/chat_template.json "HTTP/1.1 404 Not Found"
2026-05-14 11:26:06,342 - INFO - HTTP Request: HEAD https://huggingface.co/openai/whisper-tiny/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"
2026-05-14 11:26:06,450 - INFO - HTTP Request: HEAD https://huggingface.co/openai/whisper-tiny/resolve/main/audio_tokenizer_config.json "HTTP/1.1 404 Not Found"
2026-05-14 11:26:06,567 - INFO - HTTP Request: HEAD https://huggingface.co/openai/whisper-tiny/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-05-14 11:26:06,673 - INFO - HTTP Request: HEAD https://huggingface.co/openai/whisper-tiny/resolve/main/preprocessor_config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-14 11:26:06,685 - INFO - HTTP Request: HEAD https

Loading model openai/whisper-tiny...


2026-05-14 11:26:07,805 - INFO - HTTP Request: HEAD https://huggingface.co/openai/whisper-tiny/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-14 11:26:07,815 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/openai/whisper-tiny/169d4a4341b33bc18d8881c4b69c2e104e1cc0af/config.json "HTTP/1.1 200 OK"
2026-05-14 11:26:07,927 - INFO - HTTP Request: HEAD https://huggingface.co/openai/whisper-tiny/resolve/main/model.safetensors "HTTP/1.1 302 Found"


Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

2026-05-14 11:26:08,282 - INFO - HTTP Request: HEAD https://huggingface.co/openai/whisper-tiny/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-14 11:26:08,292 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/openai/whisper-tiny/169d4a4341b33bc18d8881c4b69c2e104e1cc0af/generation_config.json "HTTP/1.1 200 OK"


PEFT disabled. Training full parameters.


In [12]:
# @title Data Collator & Evaluation Metrics
import evaluate
from dataclasses import dataclass
from typing import Any, Dict, List, Union
from transformers import EvalPrediction

# Load Word Error Rate (WER) metric
wer_metric = evaluate.load("wer")

def compute_metrics(pred: EvalPrediction) -> Dict[str, float]:
    """Computes Word Error Rate (WER) for the model predictions."""
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # replace -100 with the pad_token_id
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    # Decode prediction and label ids back to strings
    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    # Compute Word Error Rate (WER)
    wer = 100 * wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    """
    Data collator that dynamically pads input features and tokenized labels,
    stacking individual dataset examples into unified batches to pass downstream to the trainer.
    """
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # Whisper processor has already padded the audio inputs to 30s (3000 frames)
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # Pad labels to max length in the current batch
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # Replace pad token id by -100 so that CrossEntropyLoss ignores padding when computing loss
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # Set target labels
        batch["labels"] = labels
        return batch

# Initialize data collator
data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

In [13]:
# Initialize GCP Storage Client
storage_client = None
if GCP_PROJECT_ID and GCP_PROJECT_ID != "<YOUR_GCP_PROJECT_ID>":
    print(f"Initializing GCP Storage Client for project {GCP_PROJECT_ID}...")
    storage_client = storage.Client(project=GCP_PROJECT_ID)

# Initialize NemoDatasets
print("Loading training dataset...")
train_dataset = NemoDataset(
    manifest_path_or_uri=TRAIN_MANIFEST_PATH,
    processor=processor,
    storage_client=storage_client
)

print("Loading validation dataset...")
val_dataset = NemoDataset(
    manifest_path_or_uri=VAL_MANIFEST_PATH,
    processor=processor,
    storage_client=storage_client
)


Initializing GCP Storage Client for project automatic-hawk-481415-m9...
Loading training dataset...
Loaded 6734 entries from manifest: gs://wd-transcription-data/segmented_audio/echo/eval/audio_raw/batch_manifest.jsonl
Loading validation dataset...
Loaded 702 entries from manifest: gs://wd-transcription-data/segmented_audio/one_hour_pilot_audio_raw/batch_manifest.jsonl


In [17]:
# @title Dataset Loading & Trainer Initialization
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    max_steps=MAX_STEPS,
    # TODO(varun): review this
    gradient_checkpointing=True,
    # This is causing issues if set to True right now
    fp16=False,
    eval_strategy="steps",
    per_device_eval_batch_size=BATCH_SIZE,
    predict_with_generate=True,
    # TODO(varun): Review -- what about training?
    generation_max_length=225,
    save_steps=SAVE_STEPS,
    eval_steps=EVAL_STEPS,
    logging_steps=LOGGING_STEPS,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=False,
    remove_unused_columns=False, # crucial since our custom Dataset doesn't match the signature columns
)

# Initialize Seq2SeqTrainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor
)

print("Seq2SeqTrainer initialized successfully.")

Seq2SeqTrainer initialized successfully.


In [18]:
# @title Run Fine-Tuning
print("Starting training...")
trainer.train()

# Save the final fine-tuned model locally
print("Training completed. Saving fine-tuned model...")
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print(f"Model and processor successfully saved to {OUTPUT_DIR}.")

Starting training...


/opt/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss,Wer
2,No log,6.177631,106.545579


[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its para

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


Training completed. Saving fine-tuned model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and processor successfully saved to ../trained_checkpoints/.


In [20]:
# @title Sync model checkpoints to GCS (Optional)
# Change GCS_OUTPUT_BUCKET and GCS_OUTPUT_PREFIX to your desired GCS target
GCS_OUTPUT_BUCKET = "<YOUR_GCS_BUCKET_NAME>"
GCS_OUTPUT_PREFIX = "whisper-finetuned-checkpoints"

if GCS_OUTPUT_BUCKET and GCS_OUTPUT_BUCKET != "<YOUR_GCS_BUCKET_NAME>" and storage_client:
    import glob
    print(f"Uploading fine-tuned model in {OUTPUT_DIR} to gs://{GCS_OUTPUT_BUCKET}/{GCS_OUTPUT_PREFIX}...")
    bucket = storage_client.bucket(GCS_OUTPUT_BUCKET)
    
    # Search for saved files under OUTPUT_DIR
    for local_filepath in glob.glob(os.path.join(OUTPUT_DIR, "**/*"), recursive=True):
        if os.path.isfile(local_filepath):
            rel_path = os.path.relpath(local_filepath, OUTPUT_DIR)
            blob_path = os.path.join(GCS_OUTPUT_PREFIX, rel_path)
            print(f"Uploading {local_filepath} to gs://{GCS_OUTPUT_BUCKET}/{blob_path}...")
            blob = bucket.blob(blob_path)
            blob.upload_from_filename(local_filepath)
    print("Finished uploading to GCS.")
else:
    print("GCS bucket not configured or storage client not initialized. Skipping GCS upload.")

GCS bucket not configured or storage client not initialized. Skipping GCS upload.


In [ ]:
# @title Inference / Validation on Test Example
from transformers import pipeline

print(f"Loading pipeline from fine-tuned weights: {OUTPUT_DIR}...")
# We can load the fine-tuned PEFT/LoRA weights directly or use the base model with PEFT weights.
# HF pipeline handles PEFT model natively if PEFT is installed!
pipe = pipeline(
    "automatic-speech-recognition",
    model=OUTPUT_DIR,
    device="cuda" if torch.cuda.is_available() else "cpu",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
)

# Test entry
TEST_AUDIO_PATH = "" # Provide a local path to a WAV/FLAC file
if TEST_AUDIO_PATH:
    print(f"Running inference on {TEST_AUDIO_PATH}...")
    result = pipe(
        TEST_AUDIO_PATH,
        generate_kwargs={
            "temperature": 0.0,
            "no_repeat_ngram_size": 3,
        }
    )
    print("Transcription result:")
    print(result["text"])
else:
    print("Provide a TEST_AUDIO_PATH to verify inference works.")